In [1]:
import sys
sys.path.append("../")  

import json
import numpy as np
import jax
import jax.numpy as jnp
import optax
    

In [2]:
TRANSFORMER_OUT_FILE = '../nogit_circuitdata/aug_angle_predictor_1022_start_subcircuits_gb10_20251028_093406_len10_enumall_100K_Nplus3.jsonl'
transformer_circuit_pqc_data = []
for line in open(TRANSFORMER_OUT_FILE):
    data = list(json.loads(line)['circuit_tokens'])
    pqc_angles = []
    base_circuit = []
    pqc_circuit = []
    for item in data:
        gate_data = item.split(':')
        if len(gate_data) != 2:
            base_circuit.append((gate_data[0], [0], []))
            pqc_circuit.append((gate_data[0], [0], []))
        else:
            pqc_angles.append(float(gate_data[1]))
            pqc_circuit.append((gate_data[0], [], [float(gate_data[1])]))


    transformer_circuit_pqc_data.append((base_circuit , pqc_circuit, pqc_angles))



In [3]:
from pqcqec.noise.simple_noise import PennylaneNoisyGates
from pqcqec.models.pqc_models import ZXZInterleavedAngleCustomStatevecModel
from pqcqec.training.jax_train_functions import train_lel_zz_custom_statevec_no_uncomp
from pqcqec.training.jax_loss_functions import jax_pure_state_fidelity
from pqcqec.simulate.simulate import get_input_data
from pqcqec.simulate.jax_statevector import build_jax_circuit, jax_run_many_states
from pqcqec.utils.jax_utils import JAXStateMeasuredDataset, JAXDataLoader
from pqcqec.utils.quaternions_utils import zxz_to_quaternion_direct

NUM_QUBITS = 1
NUM_GATES = 10
GATE_BLOCKS = 10

NUM_DATA = 1000
NUM_TEST = 100
BATCH = 10
EPOCHS = 5


In [ ]:
fine_tuning_result_list = []

for i, (base_circuit, pqc_circuit, init_pqc_angles) in enumerate(transformer_circuit_pqc_data):

    """
    Run the full experiment with custom Numba statevector backend.
    
    This uses LELZZInterleavedQuaternionCustomStatevecModel for fast simulation
    with the custom Numba backend while maintaining JAX/Optax training.
    """
    # Set random seed for reproducibility
    jax_prng_keys = jax.random.split(jax.random.PRNGKey(i), 3).flatten() # Split gives us (3,2) shape, flatten to (6,) 
    print(f"Using Seed and JAX PRNG Keys: {i, jax_prng_keys}")
    
    # Generate ideal training data (input states)
    ideal_train_data = get_input_data(NUM_QUBITS, NUM_DATA, seed=jax_prng_keys[0])
    
    noise_model = PennylaneNoisyGates(seed=jax_prng_keys[1])
    
    # Single Qubit noise arrays extracted from noise model
    np.random.seed(i)
    x_noise_arr = np.random.uniform(noise_model.x_noise_min, noise_model.x_noise_max, 
                                    (NUM_GATES,)).astype(np.float32)
    z_noise_arr = np.random.uniform(noise_model.z_noise_min, noise_model.z_noise_max, 
                                    (NUM_GATES,)).astype(np.float32)

    print(f"X-noise range: [{x_noise_arr.min():.4f}, {x_noise_arr.max():.4f}]")
    print(f"Z-noise range: [{z_noise_arr.min():.4f}, {z_noise_arr.max():.4f}]")

    # # Generate random circuit
    # qiskit_random_circuit = generate_random_circuit(
    #     num_qubits=num_qubits,
    #     num_gates=num_gates,
    #     gate_dist=gate_dist,
    #     seed=seed
    # )

    # if add_uncomputation:
    #     print("Using Uncomputation (U U†)")
    #     qiskit_adjoint_circuit = qiskit_random_circuit.inverse()
    #     qiskit_uncomp_circuit = qiskit_random_circuit.compose(qiskit_adjoint_circuit)
    #     # Double the noise arrays for the adjoint circuit
    #     x_noise_arr = np.concatenate([x_noise_arr, x_noise_arr])
    #     z_noise_arr = np.concatenate([z_noise_arr, z_noise_arr])
    # else:
    #     print("Not using Uncomputation")
    #     qiskit_uncomp_circuit = qiskit_random_circuit
        

    uncomp_circuit_ops = base_circuit
    print(f"Circuit has {len(uncomp_circuit_ops)} operations")

    # Initialize model with custom statevector backend
    # model = LELZZInterleavedQuaternionCustomStatevecModel(
    #     base_circuit_ops=uncomp_circuit_ops,
    #     num_qubits=num_qubits,
    #     x_noise=x_noise_arr,
    #     z_noise=z_noise_arr,
    #     pqc_blocks=pqc_blocks,
    #     gate_blocks=gate_blocks,
    #     seed=jax_prng_keys[4]
    # )
    model = ZXZInterleavedAngleCustomStatevecModel(
        base_circuit_ops=uncomp_circuit_ops,
        num_qubits=NUM_QUBITS,
        x_noise=x_noise_arr,
        z_noise=z_noise_arr,
        pqc_blocks=1,
        gate_blocks=GATE_BLOCKS,
        seed=jax_prng_keys[4]
    )
    init_pqc_angles = jnp.array(init_pqc_angles, dtype=jnp.float32).reshape(1,1,3)
    model.set_model_params(new_params={'pre_angles':init_pqc_angles})
    params = model.get_model_params_to_store()
    total_params = sum([p.size for p in params.values()])
    print(f"Model initialized with {total_params} trainable parameters")


    for key in params:
        print(f"  {key}: {params[key].shape}")

    # Create dataset and dataloader
    print("Generating ideal target states for training...")
    base_jax_ops = build_jax_circuit(uncomp_circuit_ops)
    ideal_train_outputs = jax_run_many_states(NUM_QUBITS, *base_jax_ops, ideal_train_data)  


    train_dataset = JAXStateMeasuredDataset(ideal_train_data, ideal_train_outputs)
    train_dataloader = JAXDataLoader(train_dataset, batch_size=BATCH, shuffle=True, 
                                    seed=jax_prng_keys[2])

    # Define optimizer with learning rate schedule
    TOTAL_STEPS = int(NUM_DATA / BATCH)
    WARMUP_STEPS = int(0.1 * TOTAL_STEPS)
    RESTART_PERIOD = int(0.25 * TOTAL_STEPS)

    INIT_LR = 1e-4
    PEAK_LR = 5e-3
    MIN_LR = 5e-5

    # 1. Warmup schedule
    warmup = optax.linear_schedule(
        init_value=INIT_LR,
        end_value=PEAK_LR,
        transition_steps=WARMUP_STEPS
    )

    # 2. Cosine decay with restarts
    def cosine_with_restart_schedule(step):
        step_in_period = step % RESTART_PERIOD
        cosine = 0.5 * (1 + jnp.cos(jnp.pi * step_in_period / RESTART_PERIOD))
        return MIN_LR + (PEAK_LR - MIN_LR) * cosine

    # 3. Stitch warmup + cosine
    schedule = optax.join_schedules(
        schedules=[warmup, cosine_with_restart_schedule],
        boundaries=[WARMUP_STEPS]
    )

    # 4. Optimizer chain
    optimizer = optax.chain(
        optax.clip_by_global_norm(1.0),
        optax.scale_by_adam(eps=1e-8),
        optax.add_decayed_weights(weight_decay=1e-5),
        optax.scale_by_schedule(schedule),
        optax.scale(-1.0)
    )
    
    print(f"\nStarting training with {EPOCHS} epochs...")
      
    train_lel_zz_custom_statevec_no_uncomp(model, train_dataloader, 
                                               optimizer, schedule, epochs=EPOCHS)

    # Test the model
    print(f"\nGenerating test data...")
    ideal_test_input_data = get_input_data(NUM_QUBITS, NUM_TEST, seed=jax_prng_keys[5])

    print(f'Ideal Test Data Shape: {ideal_test_input_data.shape}')
    
    # Generate noisy outputs using custom backend for comparison
    print(f'Running circuit with noise using custom backend on test data...')
    test_circuit_with_noise_ops = uncomp_circuit_ops.copy()
    # Add noise gates to circuit
    noisy_test_ops = []
    for i, op in enumerate(test_circuit_with_noise_ops):
        noisy_test_ops.append(op)
        gate, qubits, params = op
        for q in qubits:
            noisy_test_ops.append(('rx', [q], [float(x_noise_arr[min(i, len(x_noise_arr)-1)])]))
            noisy_test_ops.append(('rz', [q], [float(z_noise_arr[min(i, len(z_noise_arr)-1)])]))
    
    noisy_test_jax_ops = build_jax_circuit(noisy_test_ops)
    noisy_state = jax_run_many_states(NUM_QUBITS, *noisy_test_jax_ops, ideal_test_input_data)

    # Determine ideal output state
    print(f'Generating ideal (noiseless) output states...')
    base_test_jax_ops = build_jax_circuit(uncomp_circuit_ops)
    ideal_out_state = jax_run_many_states(NUM_QUBITS, *base_test_jax_ops, ideal_test_input_data)


    print(f'Running PQC model on test data...')
    pqc_state = model.run_model_batch(ideal_test_input_data)
    
    # Compute fidelities
    batched_fidelity = jax.vmap(jax_pure_state_fidelity, in_axes=(0, 0))    
    fidelity_ideal_noisy = batched_fidelity(ideal_out_state, noisy_state)
    fidelity_ideal_pqc = batched_fidelity(ideal_out_state, pqc_state)

    print(f"\n=== Test Results ===")
    print(f"Fidelity (Ideal, Noisy): {jnp.mean(fidelity_ideal_noisy):.4e} ± {jnp.std(fidelity_ideal_noisy):.4e}")
    print(f"Fidelity (Ideal, PQC): {jnp.mean(fidelity_ideal_pqc):.4e} ± {jnp.std(fidelity_ideal_pqc):.4e}")


    
    # if return_fidelity:
    #     return fidelity_ideal_noisy, fidelity_ideal_pqc

    # return uncomp_circuit_ops, model.get_circuit_tokens(), jnp.mean(fidelity_ideal_pqc).item(), model.get_pqc_params()

    # Generate noisy outputs using custom backend for comparison
    print(f'Running transformer circuit with noise using custom backend on test data...')
    # Add noise gates to circuit
    noisy_transformer_ops = []
    for i, op in enumerate(pqc_circuit):
        noisy_transformer_ops.append(op)
        gate, qubits, params = op
        if gate in ['rx', 'rz']:
            continue  # No noise after PQC gates
        for q in qubits:
            noisy_transformer_ops.append(('rx', [q], [float(x_noise_arr[min(i, len(x_noise_arr)-1)])]))
            noisy_transformer_ops.append(('rz', [q], [float(z_noise_arr[min(i, len(z_noise_arr)-1)])]))

    noisy_transformer_jax_ops = build_jax_circuit(noisy_transformer_ops)
    noisy_transformer_state = jax_run_many_states(NUM_QUBITS, *noisy_transformer_jax_ops, ideal_test_input_data)
    fidelity_ideal_transformer = batched_fidelity(ideal_out_state, noisy_transformer_state)




    

Using Seed and JAX PRNG Keys: (0, Array([1797259609, 2579123966,  928981903, 3453687069, 4146024105,
       2718843009], dtype=uint32))
X-noise range: [0.3142, 0.3142]
Z-noise range: [0.3142, 0.3142]
Circuit has 10 operations
Model initialized with 3 trainable parameters
  pre_angles: (1, 1, 3)
Generating ideal target states for training...

Starting training with 5 epochs...
Epoch 1/5


Epoch 1 summary - Mean Fidelity: 9.9832e-01, Mean Loss: 1.2429e-03
Epoch 2/5


Epoch 2 summary - Mean Fidelity: 1.0000e+00, Mean Loss: 9.0814e-07
Epoch 3/5


Epoch 3 summary - Mean Fidelity: 1.0000e+00, Mean Loss: 8.4639e-09
Epoch 4/5


Epoch 4 summary - Mean Fidelity: 1.0000e+00, Mean Loss: 4.4107e-09
Epoch 5/5


...

Epoch 4 summary - Mean Fidelity: 1.0000e+00, Mean Loss: 5.4836e-09
Epoch 5/5


Epoch 5 summary - Mean Fidelity: 1.0000e+00, Mean Loss: 5.8413e-09

Generating test data...
Ideal Test Data Shape: (100, 2)
Running circuit with noise using custom backend on test data...
Generating ideal (noiseless) output states...
Running PQC model on test data...

=== Test Results ===
Fidelity (Ideal, Noisy): 3.9965e-01 ± 3.1128e-01
Fidelity (Ideal, PQC): 1.0000e+00 ± 2.0648e-08
Using Seed and JAX PRNG Keys: (17, Array([1410583977,  344060510,  677216225, 3396477011,  853908896,
       4142250837], dtype=uint32))
X-noise range: [0.3142, 0.3142]
Z-noise range: [0.3142, 0.3142]
Circuit has 10 operations
Model initialized with 3 trainable parameters
  pre_angles: (1, 1, 3)
Generating ideal target states for training...

Starting training with 5 epochs...
Epoch 1/5


Epoch 1 summary - Mean Fidelity: 9.9983e-01, Mean Loss: 1.2561e-04
Epoch 2/5


Epoch 2 summary - Mean Fidelity: 1.0000e+00, Mean Loss: 2.2852e-07
Epoch 3/5


KeyboardInterrupt: 